In [ ]:
import os
import re
import csv
import time
import requests
from bs4 import BeautifulSoup

# -----------------------------
# CONFIGURATION
# -----------------------------
INPUT_FILE = "/content/links for military data.txt"
OUTPUT_CSV = "military_raw_data.csv"
HTML_DEBUG_DIR = "html_debug"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

os.makedirs(HTML_DEBUG_DIR, exist_ok=True)

# -----------------------------
# STEP 1: READ & CLEAN URLS
# -----------------------------
def load_clean_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    raw_urls = re.findall(r'https?://[^\s]+', text)
    clean_urls = []

    for url in raw_urls:
        url = url.strip(" '\":,")
        if url.startswith("http://"):
            url = url.replace("http://", "https://")
        clean_urls.append(url)

    return list(set(clean_urls))


# -----------------------------
# STEP 2: SCRAPE SINGLE PAGE
# -----------------------------
def scrape_page(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
    except Exception:
        return None, None

    if response.status_code != 200:
        return None, None

    # Save raw HTML for debugging
    filename = url.split("/")[-1].replace(".php", "") + ".html"
    with open(os.path.join(HTML_DEBUG_DIR, filename), "w", encoding="utf-8") as f:
        f.write(response.text)

    soup = BeautifulSoup(response.text, "html.parser")

    extracted_rows = []

    for div in soup.find_all("div"):
        text = div.get_text(" ", strip=True)

        if text and any(char.isdigit() for char in text) and len(text) < 180:
            extracted_rows.append(text)

    return extracted_rows, response.url


# -----------------------------
# STEP 3: MAIN EXECUTION
# -----------------------------
def main():
    urls = load_clean_urls(INPUT_FILE)

    print(f"Total URLs Loaded: {len(urls)}")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["country", "metric_name", "metric_value", "source_url"])

        success = 0

        for url in urls:
            print(f"Scraping: {url}")
            rows, final_url = scrape_page(url)

            if not rows:
                print("  ❌ Failed or no data")
                continue

            success += 1

            for row in rows:
                parts = row.split()
                if len(parts) < 2:
                    continue

                country = parts[0]
                metric_value = parts[-1]
                metric_name = " ".join(parts[1:-1])

                writer.writerow([country, metric_name, metric_value, final_url])

            time.sleep(1)  # polite scraping

        print(f"\nSuccessful URLs: {success}/{len(urls)}")
        print("Scraping completed.")


if __name__ == "__main__":
    main()


Total URLs Loaded: 55
Scraping: https://www.globalfirepower.com/navy-aircraft-carriers.php
Scraping: https://www.globalfirepower.com/manpower-fit-for-military-service.php
Scraping: https://www.globalfirepower.com/armor-apc-total.php
Scraping: https://www.globalfirepower.com/available-military-manpower.php
Scraping: https://www.globalfirepower.com/armor-tanks-total.php
Scraping: https://www.globalfirepower.com/roadway-coverage.php
Scraping: https://www.globalfirepower.com/oil-consumption-by-country.php
Scraping: https://www.globalfirepower.com/aircraft-total.php
Scraping: https://www.globalfirepower.com/active-reserve-military-manpower.php
Scraping: https://www.globalfirepower.com/merchant-marine-strength-by-country.php
Scraping: https://www.globalfirepower.com/coal-consumption-by-country.php
Scraping: https://www.globalfirepower.com/countries-listing.php
Scraping: https://www.globalfirepower.com/natural-gas-consumption-by-country.php
Scraping: https://www.globalfirepower.com/external-d